# The coupled spatio-sequential model: simulate, enumerate, fit

One pass through the fourth problem class (issue #290): a Potts prior over
class labels on a spatial graph, one hidden Markov chain per class, and a
*gated* emission that lets the class a node belongs to say which chain emits
its observations. Every number printed here is pinned by the regression suite
(`tests/regression/{sim,likelihood,search}/test_spatio_sequential*.py`); this
notebook states nothing the suite does not.

**Scope.** The canonical instance is small enough to enumerate --- a 2x2 open
lattice, two classes, two states, six positions, 65,536 joint states --- so
the evidence, the posteriors and the MAP labelling have exact values. Past
enumeration a planted 10x10 lattice is where the label solvers and the
annealed start are measured. Categorical emissions throughout; the Gaussian
and count families the ticket names are one constructor away and are not
shown.

In [1]:
import numpy as np
from snakes_and_ladders.likelihood.spatio_sequential import (
    enumerate_spatio_sequential,
    log_evidence_by_forward,
    map_labelling,
)
from snakes_and_ladders.opt.schedule import Exponential
from snakes_and_ladders.search.spatio_sequential import (
    LabelSolver,
    fit_spatio_sequential,
    graph_burn_in,
    label_accuracy,
)
from snakes_and_ladders.sim.fixtures import fixture
from snakes_and_ladders.sim.spatio_sequential import simulate_spatio_sequential

np.set_printoptions(precision=3, suppress=True)

## The fixture

The CI fixture of the coupled problem declares the truth. One generator draws the
labels from the Potts prior by the heat bath, each class's chain from its
initial distribution and the circulant transition, and every observation from
the family of the node's class at the state of that class's chain.

In [2]:
params = fixture("spatio_sequential", "ci").params
data = simulate_spatio_sequential(params, np.random.default_rng(1))
print(
    "nodes",
    params.graph.n_nodes,
    "classes",
    params.n_classes,
    "states",
    params.n_states,
    "positions",
    params.n_positions,
)
print("labels     ", data.labels)
print("chains     ", data.states.tolist())
print("observations (positions x nodes)")
print(data.observations)

nodes 4 classes 2 states 2 positions 6
labels      [1 1 1 0]
chains      [[0, 0, 0, 1, 1, 0], [0, 1, 1, 0, 1, 1]]
observations (positions x nodes)
[[2 0 1 0]
 [1 1 0 0]
 [1 2 1 0]
 [2 2 2 1]
 [1 1 1 1]
 [2 1 0 0]]


## The oracle, two ways

Enumeration sums the joint over every labelling and every joint chain path.
It is pinned against a route that shares no code with it: given the labels
the classes decouple, so the evidence is a sum over labellings of the
per-class forward recursion.

In [3]:
exact = enumerate_spatio_sequential(params, data.observations)
forward = log_evidence_by_forward(params, data.observations)
print("log evidence by enumeration", round(exact.log_evidence, 10))
print("log evidence by forward     ", round(forward, 10))
print("relative gap                ", abs(exact.log_evidence - forward) / abs(forward))
print("label posterior p(l_n = m | x)")
print(exact.label_posterior)
print("MAP labelling", map_labelling(params, data.observations), " drawn", data.labels)

log evidence by enumeration -27.3318388129
log evidence by forward      -27.3318388129
relative gap                 0.0
label posterior p(l_n = m | x)
[[0.067 0.933]
 [0.097 0.903]
 [0.168 0.832]
 [0.727 0.273]]
MAP labelling [1 1 1 0]  drawn [1 1 1 0]


## The label block against enumeration

With the parameters held at the truth, block ascent on the labels is a
coordinate ascent on `log p(x, l | theta)`. From the planted labels each of the
three solvers reaches the enumerated MAP labelling on most draws; the joint
never decreases.

In [4]:
schedule = Exponential(2.0, 0.2, 12)
for solver in LabelSolver:
    hits = 0
    for seed in range(6):
        draw = simulate_spatio_sequential(params, np.random.default_rng(10 + seed))
        target = map_labelling(params, draw.observations)
        fit = fit_spatio_sequential(
            params,
            draw.observations,
            np.random.default_rng(seed),
            solver=solver,
            n_blocks=6,
            labels=draw.labels,
            fit_parameters=False,
            wolff_schedule=schedule,
        )
        assert (np.diff(fit.log_likelihoods) >= -1e-9).all()
        hits += int(np.array_equal(fit.labels, target))
    print(
        f"{solver.value:16s} reaches the enumerated MAP from the planted labels on {hits} of 6 draws"
    )

alpha_expansion  reaches the enumerated MAP from the planted labels on 5 of 6 draws


icm              reaches the enumerated MAP from the planted labels on 5 of 6 draws


wolff            reaches the enumerated MAP from the planted labels on 5 of 6 draws


## Block ascent with the parameters fitted

Now the emissions, the initial distributions and the shared self-transition
are re-estimated each block. The printed trajectory is `log p(x, l | theta)`
up to the Potts normalizer after every half block, and it does not decrease.

In [5]:
fit = fit_spatio_sequential(
    params, data.observations, np.random.default_rng(0), n_blocks=8
)
print("joint after every half block")
print(fit.log_likelihoods)
print("non-decreasing:", bool((np.diff(fit.log_likelihoods) >= -1e-9).all()))
print(
    "fitted self-transition",
    round(fit.params.self_transition, 4),
    " true",
    params.self_transition,
)
print("fitted labels", fit.labels, " drawn", data.labels)

joint after every half block
[-23.838 -20.107 -20.107 -19.979 -19.979 -19.942 -19.942 -19.926 -19.926
 -19.917 -19.917 -19.912 -19.912 -19.908 -19.908 -19.906 -19.906]
non-decreasing: True
fitted self-transition 0.6038  true 0.7
fitted labels [1 1 1 0]  drawn [1 1 1 0]


## Past enumeration: the three solvers, and the start

A planted 10x10 lattice with weak emissions, so the prior matters. With the
parameters known the label problem is easy. From a uniform start with the
true parameters as EM's starting point, every solver freezes --- the trap is
the parameters, not the labels --- and the annealed start `Graph_BurnIn++`,
seeded by `Emission_Mixture++`, is what recovers them.

In [6]:
lattice = fixture("spatio_sequential", "stress").params
side = lattice.graph.shape[0]
planted = (np.arange(side * side) % side < side // 2).astype(np.int64)
rows = {
    "known parameters": [],
    **{solver.value: [] for solver in LabelSolver},
    "burn-in start": [],
}
for seed in range(6):
    draw = simulate_spatio_sequential(
        lattice, np.random.default_rng(50 + seed), labels=planted
    )
    known = fit_spatio_sequential(
        lattice,
        draw.observations,
        np.random.default_rng(seed),
        n_blocks=10,
        labels=planted,
        fit_parameters=False,
    )
    rows["known parameters"].append(label_accuracy(known.labels, planted, 2))
    for solver in LabelSolver:
        cold = fit_spatio_sequential(
            lattice,
            draw.observations,
            np.random.default_rng(seed),
            solver=solver,
            n_blocks=10,
            wolff_schedule=Exponential(3.0, 0.3, 30),
        )
        rows[solver.value].append(label_accuracy(cold.labels, planted, 2))
    warm = graph_burn_in(
        lattice,
        draw.observations,
        np.random.default_rng(seed),
        Exponential(4.0, 1.0, 30),
    )
    polished = fit_spatio_sequential(
        warm.params,
        draw.observations,
        np.random.default_rng(seed),
        n_blocks=10,
        labels=warm.labels,
    )
    rows["burn-in start"].append(label_accuracy(polished.labels, planted, 2))
print("label accuracy up to permutation, mean over six planted draws")
for name, values in rows.items():
    print(f"  {name:18s} {np.mean(values):.3f}")

label accuracy up to permutation, mean over six planted draws
  known parameters   0.987
  alpha_expansion    0.662
  icm                0.783
  wolff              0.683
  burn-in start      0.973


## Further work

- **Gaussian and count emissions in this notebook (#290).** The families exist
  and the M-step identity is pinned on a Gaussian instance in the suite; the
  notebook shows the categorical case only.
- **A larger and more frustrated lattice (#290).** The 10x10 planted instance
  separates the annealed start from a cold one; whether a cluster move ever
  beats single-site descent on a harder instance is unmeasured.
- **The comparison at an equal budget through the utility (#281).** The
  solver rows above run ten blocks each; the sweeps each label block spends
  are not held equal.
- **Decoding the chains (#407).** Viterbi landed with #175; the state
  posteriors are computed here and no path is decoded from them.
- **The textbook's coupled-model section (#298).** Its built-versus-planned
  paragraph moves once this and that land.